### Programming for Biomedical Informatics
#### Week 3 - Data Integration & Summary Analysis

Using some of the skills we've developed working with eUtils we're now going to take two different lists of genes that use different identifiers convert them to NCBI Gene IDs and then use these to merge the data together. With the final merged data we will do some calculations and plots.

In [2]:
# Preliminaries
from Bio import Entrez
import urllib.request
import json
import xml.etree.ElementTree as ET
import pandas as pd

# load my API key from the file
with open('../../bio_api_keys/ncbi.txt', 'r') as file:
    api_key = file.read().strip()

with open('../../bio_api_keys//ncbi_email.txt', 'r') as file:
    email = file.read().strip()

Entrez.api_key = api_key
Entrez.email = email

In [3]:
# Step 1 - Load the two lists that we cannot currenly combine

'''The first file contains a list of gene symbols

e.g.
GeneSymbol
ADAM10
ADAM17
APP
NAE1
APBB1
GAPDH
BACE1

The second file contains a list of RefSeq transcripts (mRNA), and their associated GO terms:

 #NC,NG,NP is refseq - rember NM is NMRNA
e.g.
RefSeqID        GOTerm  Description
NM_001320570    GO:0003824      catalytic activity # Makes protein that catalyses a reaction
NM_001320570    GO:0016787      hydrolase activity
NM_001320570    GO:0140096      catalytic activity, acting on a protein
NM_001320570    GO:0043226      organelle
NM_001320570    GO:0005634      nucleus
NM_001320570    GO:0005794      Golgi apparatus

We are going to convert Gene Symbols and Refseq IDs to NCBI Gene IDs, and then combine the two lists into a single table.
'''

# Load the gene symbols as a pandas dataframe
'''### YOUR CODE HERE ###'''
gene_df = pd.read_csv('./GeneSymbols.tsv', sep = '\t')


# Load the RefSeq data as a pandas dataframe
'''### YOUR CODE HERE ###'''
refseq_df = pd.read_csv('./transcript_functions.tsv', sep = '\t')

In [4]:
# view the first few rows of each dataframe
'''### YOUR CODE HERE ###'''
gene_df.head()

,GeneSymbol
0,ADAM10
1,ADAM17
2,APP
3,NAE1
4,APBB1


In [5]:
# view the first few rows of each dataframe
'''### YOUR CODE HERE ###'''
refseq_df.head()

,RefSeqID,GOTerm,Description
0,NM_001320570,GO:0003824,catalytic activity
1,NM_001320570,GO:0016787,hydrolase activity
2,NM_001320570,GO:0140096,"catalytic activity, acting on a protein"
3,NM_001320570,GO:0043226,organelle
4,NM_001320570,GO:0005634,nucleus


In [6]:
gene_iter = ['BRCA1', 'APP', 'CUmmmm']
gene_pd = pd.Series(gene_iter)



In [7]:
# Making auxiliary get gene id from gene symbol:

def get_gene_id(gene_iter, organism = 'Homo sapians'):
    gene_id_res = {}
    for gene in gene_iter:
        handle = Entrez.esearch(db = 'gene', term= f'{gene}[gene] AND {organism}[Organism]', retmax = 1)
        record = Entrez.read(handle)
        if record['IdList']:
            gene_id_res[gene] = record['IdList'][0]
        else:
            gene_id_res[gene] = None

    return gene_id_res



In [8]:
#Step 2 - Convert Gene Symbols to NCBI Gene IDs
esearch_params = {
    'db': 'db',
    'term': 'gene_symbol_query',
    'api_key': api_key,
    'email': email,
    'use_history': 'y'
}

# create a dictionary that maps gene symbols to gene IDs
'''### YOUR CODE HERE ###'''
gene_map_dict = get_gene_id(gene_df['GeneSymbol'])



In [11]:
# convert gene_map_dict (mapping GeneSymbol -> GeneID) into a 2-column dataframe
gene_ids_df = pd.DataFrame(list(gene_map_dict.items()), columns=['GeneSymbol', 'GeneID'])

In [12]:
gene_ids_df

,GeneSymbol,GeneID
0,ADAM10,102
1,ADAM17,6868
2,APP,351
3,NAE1,8883
4,APBB1,322
...,...,...
379,PIK3C3,5289
380,ATG2A,23130
381,ATG2B,55102
382,WIPI2,26100


## Replicate his approach



In [17]:
import urllib.parse
from numpy import record
# the base request url for eSearch
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

db = 'gene'

gene_iter = gene_df['GeneSymbol'].tolist()

db_call_string = ' OR '.join([f'{gene}[Gene]' for gene in gene_iter])
db_call_string = f'(human[Organism]) AND ({db_call_string})' #Joining in the query is () AND/OR ()

# Define the parameters for the eSearch request
# This can be nicely done using a dictionary
# Note we include the history feature of eUtils to allow us to make large queries efficiently
esearch_params = {
    'db': db,
    'term':db_call_string,
    'api_key': api_key,
    'email': email,
    'history': 'y'
}
encoded_data = urllib.parse.urlencode(esearch_params).encode('utf-8')

# make the request
request = urllib.request.Request(url, data=encoded_data)
response = urllib.request.urlopen(request)

# read into an XML object
esaerch_data_XML = ET.fromstring(response.read())

# Extract WebEnv and QueryKey
# Here we use ElementTree to extract the WebEnv and QueryKey from the XML response
# We will use these to fetch the gene ids in the next step using eSummary
webenv = esaerch_data_XML.find('WebEnv').text
query_key = esaerch_data_XML.find('QueryKey').text
count = esaerch_data_XML.find('Count').text

print('webenv:', webenv, 'query_key:', query_key, 'count:', count)

AttributeError: 'NoneType' object has no attribute 'text'

In [ ]:
query_key

NameError: name 'query_key' is not defined

In [ ]:
# The


# the base request url for eSummary
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"

['ADAM10',
 'ADAM17',
 'APP',
 'NAE1',
 'APBB1',
 'GAPDH',
 'BACE1',
 'BACE2',
 'RTN3',
 'RTN4',
 'PSENEN',
 'PSEN1',
 'PSEN2',
 'NCSTN',
 'APH1A',
 'APH1B',
 'IDE',
 'MME',
 'MAPT',
 'ND1',
 'ND2',
 'ND3',
 'ND4',
 'ND4L',
 'ND5',
 'ND6',
 'NDUFV1',
 'NDUFV2',
 'NDUFV3',
 'NDUFA1',
 'NDUFA2',
 'NDUFA3',
 'NDUFA4',
 'NDUFA4L2',
 'NDUFA5',
 'NDUFA6',
 'NDUFA7',
 'NDUFA8',
 'NDUFA9',
 'NDUFA10',
 'NDUFAB1',
 'NDUFA11',
 'NDUFA12',
 'NDUFA13',
 'NDUFB1',
 'NDUFB2',
 'NDUFB3',
 'NDUFB4',
 'NDUFB5',
 'NDUFB6',
 'NDUFB7',
 'NDUFB8',
 'NDUFB9',
 'NDUFB10',
 'NDUFB11',
 'NDUFS1',
 'NDUFS2',
 'NDUFS3',
 'NDUFS4',
 'NDUFS5',
 'NDUFS6',
 'NDUFS7',
 'NDUFS8',
 'NDUFC1',
 'NDUFC2',
 'SDHA',
 'SDHB',
 'SDHC',
 'SDHD',
 'UQCRFS1',
 'CYTB',
 'CYC1',
 'UQCRC1',
 'UQCRC2',
 'UQCRH',
 'UQCRHL',
 'UQCRB',
 'UQCRQ',
 'UQCR10',
 'UQCR11',
 'COX3',
 'COX1',
 'COX2',
 'COX4I2',
 'COX4I1',
 'COX5A',
 'COX5B',
 'COX6A1',
 'COX6A2',
 'COX6B1',
 'COX6B2',
 'COX6C',
 'COX7A1',
 'COX7A2',
 'COX7A2L',
 'COX7B',
 'CO

In [ ]:
# use eSearch to convert gene symbols to NCBI Gene IDs (for the first 10 gene symbols)
# remembering to add API key and email
# remembering to use the [Gene] field in the search
# remembering to specify human
# show the progress by printing the gene symbol and gene ID and the number of gene symbols processed so far
# This takes about 3 minutes (NB not the quickest way!)
'''### YOUR CODE HERE ###'''
eUtils_base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
eSummary = "esummary.fcgi"

# don't forget to add the API key and email
#for each id in the list pull the summary
for id in ids:
    url = f"{eUtils_base}{eSummary}?db=gene&id={id}&api_key={api_key}&email={email}"
    with urllib.request.urlopen(url) as response:
        pass

#user PrettyTable to display the results
'''### YOUR CODE HERE ###'''

In [ ]:
# Step 3 - Convert RefSeq IDs to NCBI Gene IDs

# create a dictionary that maps RefSeq IDs to Gene IDs
'''### YOUR CODE HERE ###'''

# use eSearch to convert RefSeq IDs to NCBI Gene IDs (for the first 10 RefSeq IDs)
# remembering to add API key and email
# remembering to use the [Gene] field in the search
# remembering to specify human

# Search for the gene information using the RefSeq transcript ID
'''### YOUR CODE HERE ###'''

# for the first unique 10 values in the refseq dataframe, get the gene ID
# show the progress by printing the refseq ID and gene ID and the number of refseq IDs processed so far
'''### YOUR CODE HERE ###'''

# Note, may need to encode('utf-8')

In [ ]:
#user PrettyTable to display the results
'''### YOUR CODE HERE ###'''

In [ ]:
# Step 4  - merge the refseq_to_gene_id dictionary with the refseq dataframe

# create a new column in the refseq dataframe called 'GeneID'
# fill the column with the gene IDs from the refseq_to_gene_id dictionary
'''### YOUR CODE HERE ###'''

# remove rows with missing values
'''### YOUR CODE HERE ###'''

# display the refseq dataframe
'''### YOUR CODE HERE ###'''

In [ ]:
#Step 7 - combine the gene_symbol and refseq dataframes

# convert the gene_symbol_to_id dictionary to a dataframe
'''### YOUR CODE HERE ###'''

# merge the refseq and gene_symbol_to_id_df dataframes on the 'GeneID' column
'''### YOUR CODE HERE ###'''

# drop the GeneID column
'''### YOUR CODE HERE ###'''

# display the combined dataframe
'''### YOUR CODE HERE ###'''

In [ ]:
#Step 8 - Do some basic summary analysis

# display the number of rows and columns in the combined dataframe
'''### YOUR CODE HERE ###'''

# count how many unique genes are in the combined dataframe
'''### YOUR CODE HERE ###'''

# count how many unique GO terms are in the combined dataframe
'''### YOUR CODE HERE ###'''

# display the number of unique genes and GO terms
'''### YOUR CODE HERE ###'''

# use some plots to visualise the data (up to you!)
'''### YOUR CODE HERE ###'''